# type_casting

Type casting is really about boundary conversion. Most external data arrives as strings, bytes, JSON, or mixed structures. Converting it safely is an engineering task, not a syntax trick.


## 1. Problem First

Why does this matter in real systems?

- Environment variables are strings even when they represent numbers.
- CSV files and CLI flags need explicit conversion.
- Unsafe casting creates late crashes or silent corruption.


In [ ]:
raw_timeout = "30"
timeout = int(raw_timeout)
print(timeout, type(timeout))


## 2. Minimal Concept

Core casting helpers

- `int(value)`
- `float(value)`
- `str(value)`
- `bool(value)`

The dangerous one is `bool(value)` because it uses truthiness, not semantic parsing.


## 3. Mental Model

How Python actually behaves

Casting is object construction or conversion logic. It may succeed, fail with an exception, or produce surprising values. `bool("False")` does not parse the word `False`; it only checks whether the string is empty.


In [ ]:
print(int("10"))
print(float("3.14"))
print(bool("False"))
print(bool(""))


## 4. Internal Mechanics

Constructors like `int()` and `float()` call parsing or conversion logic. They may raise `ValueError` or `TypeError`. Safe code treats casts as validation boundaries.


In [ ]:
import dis

def parse_retries(raw_value):
    return int(raw_value)

dis.dis(parse_retries)

try:
    print(int("ten"))
except ValueError as exc:
    print(type(exc).__name__, exc)


## 5. Edge Cases

Where it breaks

- `bool("False")` is `True`
- `int("3.14")` fails
- Casting `None` often raises `TypeError`
- Converting too early can destroy context for better errors


In [ ]:
print(bool("False"))

for raw in ["3.14", None]:
    try:
        print(int(raw))
    except Exception as exc:
        print(type(exc).__name__, raw)


## 6. Performance Thinking

- Simple casts are usually O(n) in input length.
- Repeated conversion in hot loops is wasteful if values can be normalized once.
- The bigger concern is correctness and error reporting.


## 7. Real Use Case

A CLI tool receives raw strings from environment variables and must build a validated runtime config.


In [ ]:
raw_config = {"port": "8080", "debug": "true"}

def parse_bool(text):
    return text.strip().lower() in {"1", "true", "yes", "on"}

config = {
    "port": int(raw_config["port"]),
    "debug": parse_bool(raw_config["debug"]),
}
print(config)


## 8. Anti-Pattern

What beginners do wrong

- Use `bool(raw_string)` for parsing boolean config.
- Cast without validation context.
- Convert values repeatedly downstream instead of once at the boundary.


In [ ]:
raw_enabled = "False"
print(bool(raw_enabled))

safe_enabled = raw_enabled.strip().lower() == "true"
print(safe_enabled)


## 9. Interview Signals

Questions FAANG asks

- Why is `bool("False")` equal to `True`?
- What exceptions can casting raise?
- Where should data conversion happen in a system?
- How would you design a safe parser for config values?


## 10. Exercise (Non-trivial)

Design a config parser for environment variables where all inputs are strings. Some keys should become integers, some booleans, some optional values, and some should remain raw text.


In [ ]:
def build_config(env):
    return {
        "port": int(env["PORT"]),
        "debug": bool(env["DEBUG"]),
    }

# Task:
# 1. Replace unsafe boolean casting.
# 2. Add validation and clear errors.
# 3. Decide what to do with missing optional keys.
